# Iris â€” train & register in MLflow

Same logic as `notebooks/train_register.py`, split into cells so it runs in
Jupyter **and** Databricks (no magics, env-var config only).

- Local: tracking goes to `sqlite:///mlruns.db` (git-ignored). Nothing is registered unless
  `MLFLOW_REGISTERED_MODEL_NAME` is set **and** you run the register cell.
- Databricks: set `MLFLOW_TRACKING_URI=databricks`, `DATABRICKS_HOST`,
  `DATABRICKS_TOKEN` via env or `dbutils.secrets` â€” never paste tokens here.
  Registered name looks like `dev.iris_prod.iris_species` (Unity Catalog).

See `.env.example` for every variable.

In [ ]:
import os

# Databricks header: widgets with the same names/defaults as
# notebooks/train_register.py. Locally (no dbutils) every lookup
# falls back to env, then the default. Secrets only via the
# kv-iris-ml-dev-7405 scope with env fallback (values never logged).


def _get_param(name: str, default: str) -> str:
    """Read a Databricks widget, else env, else default. No secrets here."""
    try:
        value = dbutils.widgets.get(name)  # noqa: F821
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name, default)


def _get_secret(scope: str, key: str, env_fallback: str = "") -> str:
    """Read a Databricks secret scope, else env, else fallback."""
    try:
        return dbutils.secrets.get(scope=scope, key=key)  # noqa: F821
    except Exception:
        return os.getenv(key, env_fallback)


try:
    dbutils.widgets.text("MLFLOW_TRACKING_URI", "sqlite:///mlruns.db")  # noqa: F821
    dbutils.widgets.text("MLFLOW_EXPERIMENT_NAME", "iris-species")  # noqa: F821
    dbutils.widgets.text("MLFLOW_REGISTERED_MODEL_NAME", "")  # noqa: F821
    dbutils.widgets.text("LLM_EXPLANATION_STYLE", "concise")  # noqa: F821
except Exception:
    pass  # local Jupyter / plain python: env vars and defaults apply

if not os.getenv("DATABRICKS_TOKEN"):
    _token = _get_secret("kv-iris-ml-dev-7405", "databricks-token")
    if _token:
        os.environ["DATABRICKS_TOKEN"] = _token
    del _token  # drop the secret from the interactive namespace right after use

In [ ]:
from pathlib import Path

import joblib
import mlflow
import mlflow.pyfunc
import pandas as pd
from dotenv import load_dotenv
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from iris_model._version import __version__
from iris_model.schema import FEATURES
from iris_model.score import score_forest
from iris_model.train import IrisPyfunc

load_dotenv()
TRACKING_URI = _get_param("MLFLOW_TRACKING_URI", "sqlite:///mlruns.db")
EXPERIMENT = _get_param("MLFLOW_EXPERIMENT_NAME", "iris-species")
REGISTERED_NAME = _get_param("MLFLOW_REGISTERED_MODEL_NAME", "")  # empty = log only
LLM_STYLE = _get_param("LLM_EXPLANATION_STYLE", "concise")
if LLM_STYLE not in ("concise", "eli5", "verbose"):
    LLM_STYLE = "concise"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)
print(f"tracking={TRACKING_URI} experiment={EXPERIMENT} code=v{__version__}")
print(f"registered_name={REGISTERED_NAME or '(log only)'} llm_style={LLM_STYLE}")

In [ ]:
bunch = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    bunch.data, bunch.target, test_size=0.2, random_state=42
)
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
acc = accuracy_score(y_test, forest.predict(X_test))
print(f"accuracy={acc:.4f}")
staging = Path("models/forest.joblib")
staging.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(forest, staging)

In [ ]:
with mlflow.start_run(run_name=f"iris-rf-v{__version__}") as run:
    mlflow.set_tags(
        {
            "project": "iris-ml",
            "env": "develop",
            "task": "notebook",
            "compute": "personal",
            "code_version": __version__,
            "llm_style": LLM_STYLE,
        }
    )
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("code_version", __version__)
    mlflow.log_metric("accuracy", float(acc))
    mlflow.log_table(
        {"feature": list(FEATURES), "importance": [float(v) for v in forest.feature_importances_]},
        "feature_importance.json",
    )
    sample_in = pd.DataFrame([dict(zip(FEATURES, X_train[0], strict=True))])
    sample_out = pd.DataFrame(score_forest(forest, sample_in.to_dict(orient="records")))
    sig = mlflow.models.infer_signature(sample_in, sample_out)
    mlflow.pyfunc.log_model(
        name="model",
        python_model=IrisPyfunc(),
        artifacts={"forest": str(staging)},
        signature=sig,
        code_paths=[str(Path("src/iris_model").resolve())],
        pip_requirements=str(Path("requirements-serving.txt").resolve()),
    )
    print("run_id:", run.info.run_id)
    RUN_ID = run.info.run_id

## Register (opt-in, future step)

Run this cell **only** when you intend to create a registry version
(requires `MLFLOW_REGISTERED_MODEL_NAME`, e.g. `dev.iris_prod.iris_species`).
Locally with the default sqlite store it creates a local registry entry â€” harmless.
Against Databricks it creates a real Unity Catalog version, so double-check
the name first.

In [ ]:
if not REGISTERED_NAME:
    print("Skipped: MLFLOW_REGISTERED_MODEL_NAME is empty (log-only mode).")
else:
    mv = mlflow.register_model(model_uri=f"runs:/{RUN_ID}/model", name=REGISTERED_NAME)
    print(f"registered {mv.name} v{mv.version} status={mv.status}")